In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data=pd.read_csv("/content/eng_-french.csv")
print(data.head())
print(data.shape)

  English words/sentences French words/sentences
0                     Hi.                 Salut!
1                    Run!                Cours !
2                    Run!               Courez !
3                    Who?                  Qui ?
4                    Wow!             Ça alors !
(175621, 2)


## Sampling of Data

In [3]:
df = data.sample(n=5000, random_state=42) # Using a random_state for reproducibility
print(df.head())
print(df.shape)

                             English words/sentences  \
2785                                    Take a seat.   
29880                           I wish Tom was here.   
53776                       How did the audition go?   
154386  I've no friend to talk to about my problems.   
149823    I really like this skirt. Can I try it on?   

                                   French words/sentences  
2785                                       Prends place !  
29880                         J'aimerais que Tom soit là.  
53776                   Comment s'est passée l'audition ?  
154386  Je n'ai pas d'ami avec lequel je puisse m'entr...  
149823    J'aime beaucoup cette jupe, puis-je l'essayer ?  
(5000, 2)


## Data preprocessing

In [4]:
import unicodedata
import re
import os
def unicode_normalize(s):
    # keep accents for French; just normalize form and remove extra spaces
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize('NFC', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def preprocess_en(s):
    if not isinstance(s, str):
        return ""
    s = unicode_normalize(s)
    s = re.sub(r'[^\w\s]', '', s)  # Remove punctuation
    s = s.lower().strip()
    return s

In [5]:
def preprocess_fr(s):
    # keep accents for French; just normalize form and remove extra spaces
    if not isinstance(s, str):
        return ""
    s = unicode_normalize(s)
    s = re.sub(r'[^\w\s]', '', s)  # Remove punctuation
    s = s.lower().strip()
    return s

df['English words/sentences'] = df['English words/sentences'].apply(preprocess_en)
df['French words/sentences'] = df['French words/sentences'].apply(preprocess_fr)

print(df.head())

                           English words/sentences  \
2785                                   take a seat   
29880                          i wish tom was here   
53776                      how did the audition go   
154386  ive no friend to talk to about my problems   
149823    i really like this skirt can i try it on   

                                   French words/sentences  
2785                                         prends place  
29880                           jaimerais que tom soit là  
53776                       comment sest passée laudition  
154386  je nai pas dami avec lequel je puisse mentrete...  
149823          jaime beaucoup cette jupe puisje lessayer  


In [6]:
df = df.rename(columns={'English words/sentences': 'english_words', 'French words/sentences': 'french_words'})
print(df.head())

                                     english_words  \
2785                                   take a seat   
29880                          i wish tom was here   
53776                      how did the audition go   
154386  ive no friend to talk to about my problems   
149823    i really like this skirt can i try it on   

                                             french_words  
2785                                         prends place  
29880                           jaimerais que tom soit là  
53776                       comment sest passée laudition  
154386  je nai pas dami avec lequel je puisse mentrete...  
149823          jaime beaucoup cette jupe puisje lessayer  


In [7]:
df.tail()

,english_words,french_words
92207,thatll be a big achievement,ce sera une grande réussite
70991,some of them are teachers,certaines dentre elles sont enseignantes
35912,ill leave you to it,je te laisse ten occuper
166752,tom wished that he could play tennis as well a...,tom espérait quil pourrait jouer au tennis aus...
129861,i want to thank those who helped me,je souhaite remercier ceux qui mont aidé


In [8]:
!pip install transformers sentencepiece

## Tokenization using Pretrained Data

In [9]:
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import create_optimizer

# Load model
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

# Prepare your dataset
# Assume df has: df['english_word'], df['french_words']
train_texts = df['english_words'].tolist()
train_labels = df['french_words'].tolist()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFMarianMTModel.

All the layers of TFMarianMTModel were ini

### Spliting of Data

In [20]:
from sklearn.model_selection import train_test_split

# Split before preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    train_texts, train_labels, test_size=0.1, random_state=42
)

In [11]:
# Tokenize
def preprocess(texts, labels, max_len=64):
    model_inputs = tokenizer(texts, max_length=max_len, truncation=True, padding="max_length", return_tensors="tf")
    with tokenizer.as_target_tokenizer():
        labels_enc = tokenizer(labels, max_length=max_len, truncation=True, padding="max_length", return_tensors="tf")
    model_inputs["labels"] = labels_enc["input_ids"]
    return model_inputs

inputs = preprocess(X_train, y_train)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [12]:
print(inputs)

{'input_ids': <tf.Tensor: shape=(4950, 64), dtype=int32, numpy=
array([[   12,   122,  1526, ..., 59513, 59513, 59513],
       [   52,    55,  5348, ..., 59513, 59513, 59513],
       [   33,  3885,    75, ..., 59513, 59513, 59513],
       ...,
       [    4,   780,  2675, ..., 59513, 59513, 59513],
       [ 8068,  1396,    55, ..., 59513, 59513, 59513],
       [  128,  1124,    12, ..., 59513, 59513, 59513]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(4950, 64), dtype=int32, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int32)>, 'labels': <tf.Tensor: shape=(4950, 64), dtype=int32, numpy=
array([[   49, 10045, 39966, ..., 59513, 59513, 59513],
       [   43,  1108, 14285, ..., 59513, 59513, 59513],
       [  463,    76,  2349, ..., 59513, 59513, 59513],
       ...,
       [    8,   780,    15, ..., 59513, 595

In [13]:
# Build tf.data.Dataset
dataset = tf.data.Dataset.from_tensor_slices((
    dict(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]),
    inputs["labels"]
))
dataset = dataset.shuffle(1000).batch(16)
dataset

<_BatchDataset element_spec=({'input_ids': TensorSpec(shape=(None, 64), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(None, 64), dtype=tf.int32, name=None)}, TensorSpec(shape=(None, 64), dtype=tf.int32, name=None))>

In [14]:
# Optimizer
steps_per_epoch = len(dataset)
num_train_steps = steps_per_epoch * 3  # 3 epochs
optimizer, schedule = create_optimizer(init_lr=5e-5, num_warmup_steps=0, num_train_steps=num_train_steps)

# Compile model
model.compile(optimizer=optimizer)

In [15]:
# Train
model.fit(dataset, epochs=3)

Epoch 1/3
310/310 [==============================] - 228s 236ms/step - loss: 0.7946
Epoch 2/3
310/310 [==============================] - 73s 237ms/step - loss: 0.4474
Epoch 3/3
310/310 [==============================] - 73s 237ms/step - loss: 0.3265


In [21]:
# Test prediction on your own text
def translate_en_to_fr(text):
    inputs = tokenizer(text, return_tensors="tf", padding=True, truncation=True)
    outputs = model.generate(**inputs, max_length=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [22]:
predictions = [translate_en_to_fr(x) for x in X_test]

In [23]:
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

smooth = SmoothingFunction().method1
references = [[ref.split()] for ref in y_test]
candidates = [pred.split() for pred in predictions]

bleu_score = corpus_bleu(references, candidates, smoothing_function=smooth)
print(f"BLEU score on test set: {bleu_score*100:.2f}")

BLEU score on test set: 75.67


In [24]:
print(translate_en_to_fr("Good morning, how are you?"))
print(translate_en_to_fr("can you give me your rice?"))

Bonjour, comment allezvous ?
pouvezvous me donner votre riz ?
